In [1]:
import sys
sys.path.append("..")

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import itertools
import tqdm
from src.algorithm import find_partitions_greedy, find_partitions_optimal
from src.metric import approximation_ratio
from src.subroutine import evaluate_system

In [3]:
np.random.seed(0)

In [4]:
x_min, x_max, x_disc = 0., 1., 1e-4
X = np.arange(x_min, x_max+x_disc, x_disc).round(4)
N_THRESHOLDS = 3

In [5]:
def generate_prior_grid(n_components=3, n_balls=50):
    step = 1.0 / n_balls
    grids = []
    for combo in itertools.combinations_with_replacement(range(n_components), n_balls):
        counts = np.bincount(combo, minlength=n_components)
        grids.append(counts * step)
    return grids

In [6]:
np.random.seed(0)

def sweep(grid):
    d = {"thresholds": [], "priors": [], "c": [], "t*": [], "partition_opt": [], "partition_greedy": [], "loss_opt": [], "loss_greedy": [], "r_mult": [], "r_add": []}
    c = 0.5
    thresholds = np.array([0.25, 0.5, 0.75])
    threshold_true = np.min(thresholds)
    # threshold_true = np.median(thresholds)
    for priors in tqdm.tqdm(grid): 
        partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
        partition_greedy = find_partitions_greedy(X, thresholds, priors, threshold_true, c)
        loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
        loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
        r_mult  = approximation_ratio(loss_opt, loss_greedy, rtype="mult")
        r_add  = approximation_ratio(loss_opt, loss_greedy, rtype="add")
        if not np.isnan(r_mult):
            d["thresholds"].append(thresholds)
            d["priors"].append(priors)
            d["c"].append(c)
            d["t*"].append(threshold_true)
            d["partition_opt"].append(partition_opt)
            d["partition_greedy"].append(partition_greedy)
            d["loss_opt"].append(loss_opt.item())
            d["loss_greedy"].append(loss_greedy.item())
            d["r_mult"].append(r_mult.item())
            d["r_add"].append(r_add.item())

    return d

In [7]:
priors_grid = generate_prior_grid(3, n_balls=100)
d = sweep(priors_grid)

100%|██████████| 5151/5151 [00:33<00:00, 154.83it/s]


In [8]:
df = pd.DataFrame(d)
i = np.argmax(d["r_mult"])
df.iloc[[i]]

,thresholds,priors,c,t*,partition_opt,partition_greedy,loss_opt,loss_greedy,r_mult,r_add
402,"[0.25, 0.5, 0.75]","[0.73, 0.03, 0.24]",0.5,0.25,"[[1], [0, 2]]","[[2], [0, 1]]",0.191229,0.247608,1.294827,0.056379


In [12]:
fig = px.scatter(df, y="r_mult")
fig.update_layout(
    width=550, height=450,
    font=dict(family="iosevka"),
    yaxis=dict(title="Approximation Ratio"),
    margin=dict(t=25,b=25,l=25,r=25)
    )
fig.update_traces(marker=dict(color="rgba(37, 99, 235, 1)"))